# Scraping data from [Cane Richardson ICME catalog](https://izw1.caltech.edu/ACE/ASC/DATA/level3/icmetable2.htm) and cleaning it. Allocating datetime format timestamps to the events.

Hi there! Some useful information to save your time: This jupyter notebook has two sections 1.`Main` 2.`essentials`. If the purpose is to only scrape into a .csv file and utilize the ICME catalog you only need `Main` section.

15:49 EDT; March 25, 2026

## Main

In [1]:
import numpy as np
import matplotlib.pyplot as plt

import pandas as pd
pd.set_option('display.max_columns', None)

import requests
from bs4 import BeautifulSoup as BS

import os
import time
import glob
import datetime

In [2]:
main_dir = os.path.dirname(os.path.dirname(os.getcwd()))
main_dir

'C:\\Users\\priya\\Work\\Projects\\SVS'

### Scraping

In [3]:
url = 'https://izw1.caltech.edu/ACE/ASC/DATA/level3/icmetable2.htm'   # link to Cane Richardson ICME catalog

response = requests.get(url)
soup = BS(response.content, 'html.parser')

In [4]:
tabular = soup.find_all('table')
headers1 = [th.text.strip() for th in tabular[0].find_all('tr')]

In [5]:
headers = headers1[0].split('\n')
headers

['Disturbance Y/M/D (UT) (a) ',
 'ICME Plasma/Field Start, End Y/M/D (UT) (b)  ',
 'Comp. Start, End (Hrs wrt. Plasma/ Field) (c) ',
 'MC Start, End (Hrs wrt. Plasma/ Field) (d) ',
 'BDE? (e) ',
 'BIF? (f) ',
 ' Qual. (g)',
 'dV (km/s) (h) ',
 'V_ICME (km/s) (i) ',
 'V_max (km/s) (j) ',
 'B (nT) (k) ',
 'MC? (l) ',
 'Dst (nT) (m) ',
 'V_transit (km/s) (n) ',
 'LASCO CME Y/M/D (UT) (o)']

In [6]:
%%time
data_entries = []
for enum, ii in enumerate(tabular[0].find_all('td')[0 : 11601]):
    if '<b>' not in str(ii):
        data_entries.append(ii)

CPU times: total: 422 ms
Wall time: 443 ms


In [7]:
data_entries

[<td>1996/05/27 1500 </td>,
 <td>1996/05/27 1500 </td>,
 <td> 1996/05/29 0300 </td>,
 <td>... </td>,
 <td>... </td>,
 <td> 0 </td>,
 <td> +4 </td>,
 <td>N </td>,
 <td>... </td>,
 <td> 2 </td>,
 <td>0  </td>,
 <td>370 </td>,
 <td>  400 </td>,
 <td>   9 </td>,
 <td>  2 </td>,
 <td>            -33 </td>,
 <td> ... </td>,
 <td> </td>,
 <td>1996/07/01 1320 </td>,
 <td>1996/07/01 1800 </td>,
 <td> 1996/07/02 1100 </td>,
 <td>... </td>,
 <td>... </td>,
 <td> 0  </td>,
 <td> 0 </td>,
 <td>N </td>,
 <td>... </td>,
 <td> 3 </td>,
 <td>  40  </td>,
 <td>360 </td>,
 <td>  370 </td>,
 <td>  11 </td>,
 <td>  2 </td>,
 <td>         -20 </td>,
 <td> ... </td>,
 <td>    </td>,
 <td>1996/08/07 0600 </td>,
 <td>1996/08/07 1200 </td>,
 <td> 1996/08/08 1000 </td>,
 <td>... </td>,
 <td>... </td>,
 <td>0 </td>,
 <td> 0 </td>,
 <td>N </td>,
 <td>... </td>,
 <td> 2 </td>,
 <td>10  </td>,
 <td>350 </td>,
 <td>  380 </td>,
 <td>   7 </td>,
 <td>  2 </td>,
 <td>         -23 </td>,
 <td> ... </td>,
 <td> </td>,
 <

#### `Note`: Not all the rows are read uniformly due to inherent nature of the .html version of the ICME catalog, leaving us with uneven number of data entries. There are some (say) ghost entries which are just empty but they occupy an index and mess up the iterative cleaning process. We manually leave them out.

### Cleaning

In [8]:
# manually extending the data entries for special cases

data_entries_edit = data_entries[0 : 596]
data_entries_edit.append(BS('<td> 1998/02/20 0000</td>}', 'html.parser').find('td'))
data_entries_edit.extend(data_entries[596 : 5129])
data_entries_edit.extend(data_entries[5130 : 5148])
data_entries_edit.extend(data_entries[5149 : 5653])
data_entries_edit.extend(data_entries[5654 : 5690])
data_entries_edit.extend(data_entries[5691 : 5709])
data_entries_edit.extend(data_entries[5710 : 5728])
data_entries_edit.extend(data_entries[5729 : 5747])
data_entries_edit.extend(data_entries[5748 : 5766])
data_entries_edit.extend(data_entries[5767 : 5785])
data_entries_edit.extend(data_entries[5786 : 5804])
data_entries_edit.extend(data_entries[5805 : 5823])
data_entries_edit.extend(data_entries[5824 : ])

In [9]:
len(data_entries) / 18, len(data_entries_edit) / 18

(619.5555555555555, 619.0)

In [10]:
%%time
data_rows = []
for ii in range(619):
    data_rows.append(data_entries_edit[ii * 18 : (ii + 1) * 18])

CPU times: total: 0 ns
Wall time: 0 ns


In [11]:
headers

['Disturbance Y/M/D (UT) (a) ',
 'ICME Plasma/Field Start, End Y/M/D (UT) (b)  ',
 'Comp. Start, End (Hrs wrt. Plasma/ Field) (c) ',
 'MC Start, End (Hrs wrt. Plasma/ Field) (d) ',
 'BDE? (e) ',
 'BIF? (f) ',
 ' Qual. (g)',
 'dV (km/s) (h) ',
 'V_ICME (km/s) (i) ',
 'V_max (km/s) (j) ',
 'B (nT) (k) ',
 'MC? (l) ',
 'Dst (nT) (m) ',
 'V_transit (km/s) (n) ',
 'LASCO CME Y/M/D (UT) (o)']

In [12]:
%%time
dataframes = []

columns = ['Disturbance [UT]', 'Quality', 'V_ICME [km/s]', 'V_max [km/s]', 'B [nT]', 'MC?', 'DsT [nT]', 'V_transit [km/s]', 'LASCO CME DateTime [UT]']

for enum, ii in enumerate(data_rows[ : 611]):
    
    data = []
    
    if 'H' in str(ii[-1]).replace('<td>', '').replace('</td>', '').strip():
        
        disturbance = str(ii[0]).replace('<td>', '').replace('</td>', '').replace('\r', '').replace('\n', '').replace('  ', ' ').strip().split(' ')
        disturbance_datesplit = disturbance[0].split('/')
        
        quality         = str(ii[9]).replace('<td>', '').replace('</td>', '').strip()
        vICME           = str(ii[11]).replace('<td>', '').replace('</td>', '').strip()
        vMAX            = str(ii[12]).replace('<td>', '').replace('</td>', '').strip()
        B               = str(ii[13]).replace('<td>', '').replace('</td>', '').strip()
        MC              = str(ii[14]).replace('<td>', '').replace('</td>', '').strip()
        Dst             = str(ii[15]).replace('<td>', '').replace('</td>', '').strip()
        vTRANSIT        = str(ii[16]).replace('<td>', '').replace('</td>', '').strip()
        LASCO_datetime  = str(ii[17]).replace('<td>', '').replace('</td>', '').replace('  ', ' ').strip().split(' ')
        LASCO_datesplit = LASCO_datetime[0].split('/')
        
        data = [
                datetime.datetime(int(disturbance_datesplit[0]), int(disturbance_datesplit[1]), int(disturbance_datesplit[2]), int(disturbance[1][0 : 2]), int(disturbance[1][2 : 4])),
                quality,
                vICME,
                vMAX,
                B,
                MC,
                Dst,
                vTRANSIT,
                datetime.datetime(int(LASCO_datesplit[0]), int(LASCO_datesplit[1]), int(LASCO_datesplit[2]), int(LASCO_datetime[1][0 : 2]), int(LASCO_datetime[1][2 : 4]))
               ]
        
        try:
            print(LASCO_datetime[2], enum)   # probe to check errors while developing
        except:
            print(enum)
       
        dataframes.append(pd.DataFrame(data = [data], columns = columns))
        
for ii in [data_rows[611]]:
    
    if 'H' in str(ii[-1]).replace('<td>', '').replace('</td>', '').strip():
        
        disturbance = str(ii[0]).replace('<td>', '').replace('</td>', '').replace('\r', '').replace('\n', '').replace('  ', ' ').strip().split(' ')
        disturbance_datesplit = disturbance[0].split('/')
        
        quality = str(ii[9]).replace('<td>', '').replace('</td>', '').strip()
        vICME = str(ii[11]).replace('<td>', '').replace('</td>', '').strip()
        vMAX = str(ii[12]).replace('<td>', '').replace('</td>', '').strip()
        B = str(ii[13]).replace('<td>', '').replace('</td>', '').strip()
        MC = str(ii[14]).replace('<td>', '').replace('</td>', '').strip()
        Dst = str(ii[15]).replace('<td>', '').replace('</td>', '').strip()
        vTRANSIT = str(ii[16]).replace('<td>', '').replace('</td>', '').strip()
        LASCO_datetime = str(ii[17]).replace('<td>', '').replace('</td>', '').replace('  ', ' ').strip().split(' ')
        LASCO_datesplit = LASCO_datetime[0].split('/')
        
        data = [
                datetime.datetime(int(disturbance_datesplit[0]), int(disturbance_datesplit[1]), int(disturbance_datesplit[2]), int(disturbance[1][0 : 2]), int(disturbance[1][2 : 4])),
                quality,
                vICME,
                vMAX,
                B,
                MC,
                Dst,
                vTRANSIT,
                datetime.datetime(2025, 5, 31, 0, 15)
               ]
        
        try:
            print(LASCO_datetime[2], 611, '----')
        except:
            print(enum)
       
        dataframes.append(pd.DataFrame(data = [data], columns = columns))

H 3
H 4
H 5
H 6
H 8
H 13
H 15
H 16
H 17
H 20
H 21
H 26
H 29
H 30
H 38
H<a 39
H 55
H 57
H 58
H 71
H 75
H 80
H 81
H 87
H 88
H 95
H 96
H 97
H 98
H 99
H 103
H 114
H 121
H 123
H 125
H 129
H 131
H 136
H 137
H 138
H 139
H 145
H 146
H 150
H 151
H 152
H 154
H 155
H 156
H 160
H 172
H 179
H 183
H 184
H 189
H 190
H 191
H 199
H 200
H 201
H 202
H 203
H 204
H 209
H 212
H 226
H 227
H? 233
H 239
H 240
H 241
H 243
H 244
H 250
H 252
H 253
H 256
H 257
H 259
H 260
H 261
H 264
H 265
H 266
H 267
H 270
H 272
H 275
285
H 297
H 298
H 304
H 306
H? 308
H 322
H 325
HDW 329
HDW 340
HD 348
HD 349
H 363
HD 366
HD 371
2012/03/04 377
HD 378
HD 379
H 382
HWD 384
H 385
HD 386
HD 387
HD 390
HD 391
HD? 393
H 394
HD 398
HD 401
HD 402
HD 409
HD 411
HD 419
HD 421
HD 424
	2013/11/08 426
HD 433
HD 437
HD 438
HD 442
HD 445
H 446
HD 447
H 449
HD 454
D, 462
HD 465
HD 466
HD 475
H 480
HD 490
HD 499
H 500
HD 502
HD 520
HD 525
HD 528
D, 529
532
D, 535
HD, 536
HD 539
HD 551
HD 555
D, 556
HD 558
HD 559
HD 560
HD,	2023/07/11 563
HD 565


### Extracting

In [13]:
dfCane = pd.concat(dataframes).reset_index(drop = True)
dfCane.insert(9, 'Multiple CMEs', False)
dfCane

,Disturbance [UT],Quality,V_ICME [km/s],V_max [km/s],B [nT],MC?,DsT [nT],V_transit [km/s],LASCO CME DateTime [UT],Multiple CMEs
0,1996-12-23 16:00:00,2,360,420,10,2,-18,435,1996-12-19 16:30:00,False
1,1997-01-10 01:04:00,1,450,460,14,2,-78,507,1997-01-06 15:10:00,False
2,1997-02-09 13:21:00,2,450,600,8,2,-68,683,1997-02-07 00:30:00,False
3,1997-04-10 17:45:00,1,460,470,20,2,-82,552,1997-04-07 14:27:00,False
4,1997-05-15 01:59:00,1,450,480,21,2,-115,616,1997-05-12 05:30:00,False
...,...,...,...,...,...,...,...,...,...,...
171,2024-10-06 07:39:00,3,480,530,13,0,-96 P,620,2024-10-03 12:48:00,False
172,2024-10-10 15:14:00,1,690,750,20,2,-333 P,730-1120,2024-10-08 06:12:00,False
173,2024-12-31 16:21:00,2,510,530,15,0,-212 P,660-900,2024-12-29 01:23:00,False
174,2025-04-15 17:21:00,1,610,610,15,1,-138 P,640-730,2025-04-13 00:12:00,False


### Special cases

Note: These are speical/anomalous entries (purely in terms of technical nature here) for my work. The following list might vary depending upon the purpose.

- for ii = 103, LASCO CME DateTime = 2012/03/04 1100 | 

- for ii = 123 (multiple CMEs)

- for ii = 133, LASCO CME DateTime = 2015/05/02 2024 | 2015/05/02 0948 D, 2024 HD

- for ii = 145, LASCO CME DateTime = 2021/11/02 0248 | 2021/11/01 0200 D, 1824 D, 2124 D, 2021/11/02 0248 HD

- for ii = 147, LASCO CME DateTime = 2022/03/09 1848 | 2022/03/09 2348 D, 0200 D, 1848 HD

- for ii = 148 (multiple HCMEs)

- for ii = 152, LASCO CME DateTime = 2023/03/20 1442 | 2023/03/20 0241 D, 1442 HD

- for ii = 156 (multiple HCMEs)

- for ii = 163, LASCO CME DateTime = 2023/12/14 1712 | 2023/12/14 D, 2023/12/14 1712 HD

- for ii = 164 (multiple CMEs)

- for ii = 166, LASCO CME DateTime = 2024/05/08 0536 | 2024/05/07 0312 D, 0524 D, 2024/05/08 0536 HD, 1224 HD, 1912 D, 2224 HD, 2024/05/09 0924 HD, 1823 HD 

- for ii = 167 (multiple CMEs)

- for ii = 168 (multiple CMEs)

- for ii = 169 (multiple CMEs)

- for ii = 172 (multiple CMEs)

- for ii = 173 (multiple CMEs)

- for ii = 174 (multiple CMEs)



In [14]:
ii = 174
dfCane[ii : ii + 1]

,Disturbance [UT],Quality,V_ICME [km/s],V_max [km/s],B [nT],MC?,DsT [nT],V_transit [km/s],LASCO CME DateTime [UT],Multiple CMEs
174,2025-04-15 17:21:00,1,610,610,15,1,-138 P,640-730,2025-04-13 00:12:00,False


### Manually inputting the data from the ICME catalog for the anomalous entries

In [15]:
ii = 103
dfCane.loc[ii, 'LASCO CME DateTime [UT]'] = datetime.datetime(2012, 3, 3, 11, 0)
dfCane.loc[ii, 'Multiple CMEs'] = True

ii = 123
dfCane.loc[ii, 'Multiple CMEs'] = True

ii = 133
dfCane.loc[ii, 'LASCO CME DateTime [UT]'] = datetime.datetime(2015, 5, 2, 20, 24)
dfCane.loc[ii, 'Multiple CMEs'] = True

ii = 145
dfCane.loc[ii, 'LASCO CME DateTime [UT]'] = datetime.datetime(2011, 11, 1, 2, 48)
dfCane.loc[ii, 'Multiple CMEs'] = True

ii = 147
dfCane.loc[ii, 'LASCO CME DateTime [UT]'] = datetime.datetime(2022, 3, 9, 18, 48)
dfCane.loc[ii, 'Multiple CMEs'] = True

ii = 148
dfCane.loc[ii, 'Multiple CMEs'] = True

ii = 152
dfCane.loc[ii, 'LASCO CME DateTime [UT]'] = datetime.datetime(2023, 3, 20, 14, 42)
dfCane.loc[ii, 'Multiple CMEs'] = True

ii = 156
dfCane.loc[ii, 'Multiple CMEs'] = True

ii = 163
dfCane.loc[ii, 'LASCO CME DateTime [UT]'] = datetime.datetime(2023, 12, 14, 17, 12)
dfCane.loc[ii, 'Multiple CMEs'] = True

ii = 164
dfCane.loc[ii, 'Multiple CMEs'] = True

ii = 166
dfCane.loc[ii, 'LASCO CME DateTime [UT]'] = datetime.datetime(2024, 5, 8, 5, 36)
dfCane.loc[ii, 'Multiple CMEs'] = True

ii = 167
dfCane.loc[ii, 'Multiple CMEs'] = True

ii = 168
dfCane.loc[ii, 'Multiple CMEs'] = True

ii = 169
dfCane.loc[ii, 'LASCO CME DateTime [UT]'] = datetime.datetime(2024, 8, 7, 14, 12)
dfCane.loc[ii, 'Multiple CMEs'] = True
dfCane.loc[ii, 'V_transit [km/s]'] = '510'

ii = 172
dfCane.loc[ii, 'LASCO CME DateTime [UT]'] = datetime.datetime(2024, 10, 8, 2, 12)
dfCane.loc[ii, 'Multiple CMEs'] = True

ii = 173
dfCane.loc[ii, 'LASCO CME DateTime [UT]'] = datetime.datetime(2024, 12, 29, 6, 12)
dfCane.loc[ii, 'Multiple CMEs'] = True

ii = 174
dfCane.loc[ii, 'LASCO CME DateTime [UT]'] = datetime.datetime(2025, 4, 13, 6, 36)
dfCane.loc[ii, 'Multiple CMEs'] = True

In [16]:
dfCane.to_csv(os.path.join(main_dir, 'data', 'dfCane_scraped_basic.csv'), index = False)

## essentials 
Go beyond this line if you want to learn how to go about web scraping from scratch. The basics and smart tips to develop the code from beginning. The code below might not be in chronological order and might break, it is only for reference. It will make sense to only those who have to do it on their own and failed.

In [ ]:
# example for a data row
headers1[2]

In [ ]:
#
headers1[3].split(' ')

In [ ]:
#
for ii in headers1[1 : 20]:
    print(len(ii.split(' ')))

In [ ]:
#
headers1[324].split(' ')

In [ ]:
#
tabular[0].find_all('td')[15 : ]

In [ ]:
#
for enum, ii in enumerate(tabular[0].find_all('td')[15 : 33]):
    print(enum, '   ', ii)

In [ ]:
#
for enum, ii in enumerate(tabular[0].find_all('td')[33 : 51]):
    print(enum, '   ', ii)

In [ ]:
#
for enum, ii in enumerate(tabular[0].find_all('td')[51 : 69]):
    print(enum, '   ', ii)


In [ ]:
#
for enum, ii in enumerate(tabular[0].find_all('td')[69 : 87]):
    print(enum, '   ', ii)

In [ ]:
#
for enum, ii in enumerate(tabular[0].find_all('td')[87 : 105]):
    print(enum, '   ', ii)

#### Deductions: 18 entries for rows and 15 entries for headers

In [ ]:
ii = 11583
tabular[0].find_all('td')[ii : ii + 18]

In [ ]:
#
tabular[0].find_all('td')[11601]

In [ ]:
#
ii = 14
print(tabular[0].find_all('td')[ii])
'<b>' not in str(tabular[0].find_all('td')[ii])

In [ ]:
ii = 33
display(data_entries[18 * ii : 18 * (ii + 1)])

In [538]:
ii = 618

print(len(data_entries_edit[18 * ii : 18 * (ii + 1)])), display(data_entries_edit[18 * ii : 18 * (ii + 1)])

18


[<td>
 
 2025/09/06 1424 </td>,
 <td> 2025/09/06 1700 </td>,
 <td> 2025/09/08 1500 </td>,
 <td>...</td>,
 <td>...</td>,
 <td>...</td>,
 <td>...</td>,
 <td>Y</td>,
 <td>...</td>,
 <td> 3</td>,
 <td>100 </td>,
 <td>  660</td>,
 <td>660</td>,
 <td> 9</td>,
 <td> 2</td>,
 <td>-45 Q</td>,
 <td> 990 </td>,
 <td>  2025/09/04 2030 D</td>]

(None, None)

In [218]:
ii = 18 * 323 + 3
data_entries[ii : ii + 18]

[<td>    430  </td>,
 <td>4 </td>,
 <td>0    </td>,
 <td>-2     </td>,
 <td>536  </td>,
 <td> 2009/12/16 0430 H  <td></td></td>,
 <td></td>,
 <td>
 2010/01/01 2200 </td>,
 <td> 2010/01/01 2200 </td>,
 <td> 2010/01/03 1000 </td>,
 <td>...</td>,
 <td>ns</td>,
 <td>...</td>,
 <td>...</td>,
 <td>Y</td>,
 <td>...</td>,
 <td> 1 </td>,
 <td>10 </td>]

In [222]:
18 * 323 + 3
data_entries[5823]

<td></td>

In [212]:
data_entries[5804]

<td></td>

In [201]:
data_entries[5785]

<td></td>

In [187]:
data_entries[5766]

<td></td>

In [138]:
data_entries[5728]

<td></td>

In [130]:
data_entries[5709]

<td></td>

In [108]:
data_entries[5689]

<td> <td></td></td>

In [166]:
data_entries[5652]

<td> <td></td></td>

In [74]:
ii = 18 * 313 + 20
data_entries[ii : ii + 19]

[<td>
 2009/02/03 2000 </td>,
 <td> 2009/02/04 0000 </td>,
 <td> 2009/02/04 1600 </td>,
 <td> 0</td>,
 <td> +5</td>,
 <td>...</td>,
 <td>...</td>,
 <td>...</td>,
 <td>...</td>,
 <td>2 </td>,
 <td>40  </td>,
 <td>360 </td>,
 <td>380  </td>,
 <td>10 </td>,
 <td>2    </td>,
 <td>-42  </td>,
 <td> ...  </td>,
 <td>  </td>,
 <td>
 2009/06/03 1600 </td>]

In [38]:
18 * 314

5652

In [578]:
18 * 285 - 1, 18 * 286 

(5129, 5148)

In [577]:
data_entries[18 * 285 - 1], data_entries[18 * 286]

(<td></td>, <td></td>)

In [219]:
str(data_entries[32 * 18 + 2])

'<td> 1998/02/17 2100</td>'

In [255]:
data_entries[595 : 596], data_entries[596 : 598]

([<td>  1998/02/19 0100</td>], [<td>-3</td>, <td>...</td>])

In [596]:
2 +2 

4

In [604]:
a = ['1', '2']
print(a)

a.append('a')
print(a)

['1', '2']
['1', '2', 'a']


In [598]:
a = ['1']
c = a.append('0')
print(c)

None


In [638]:
a = ['1', 'a']
b = ['2', 'b']
c = ['3']

c.append(a)
c.append(b)
b.extend(a)
print(c), print(b)

['3', ['1', 'a'], ['2', 'b', '1', 'a']]
['2', 'b', '1', 'a']


(None, None)

In [645]:
len(data_entries_edit)

619.5

In [641]:
len(data_entries_edit)

11151

In [232]:
ii = 33
for ii in data_entries[ii * 18 : (ii + 1) * 18]:
    display(ii), display(str(ii)), print('\n')

<td>
 1998/02/18 0750(A)</td>

'<td>\r\n 1998/02/18 0750(A)</td>'

<td>  1998/02/19 0100</td>

'<td>  1998/02/19 0100</td>'

<td>-3</td>

...

'<td> ...</td>'

<td>       </td>

'<td>      \xa0</td>'

<td>
 1998/03/04 1156</td>

'<td>\r\n 1998/03/04 1156</td>'

In [230]:
33 * 18 + 1

595

In [238]:
aa = [data_entries[595]] + [BS('<td> 1998/02/20 0000</td>}', 'html.parser').find('td')] + [data_entries[596]]
aa

[<td>  1998/02/19 0100</td>, <td> 1998/02/20 0000</td>, <td>-3</td>]

In [243]:
for ii in aa:
    print(ii, type(ii))

<td>  1998/02/19 0100</td> <class 'bs4.element.Tag'>
<td> 1998/02/20 0000</td> <class 'bs4.element.Tag'>
<td>-3</td> <class 'bs4.element.Tag'>


In [224]:
BS('<td> 1998/02/20 0000</td>}', 'html.parser').find('td')

<td> 1998/02/20 0000</td>

In [ ]:
BS('<td> 1998')

In [ ]:
bs4.elm

In [203]:
data_entries[0]

<td>1996/05/27 1500 </td>

In [127]:
'\xa0' in str(data_entries[17])

True

In [147]:
data_entries[0 : 72]

[<td>1996/05/27 1500 </td>,
 <td>1996/05/27 1500 </td>,
 <td> 1996/05/29 0300 </td>,
 <td>... </td>,
 <td>... </td>,
 <td> 0 </td>,
 <td> +4 </td>,
 <td>N </td>,
 <td>... </td>,
 <td> 2 </td>,
 <td>0  </td>,
 <td>370 </td>,
 <td>  400 </td>,
 <td>   9 </td>,
 <td>  2 </td>,
 <td>            -33 </td>,
 <td> ... </td>,
 <td> </td>,
 <td>1996/07/01 1320 </td>,
 <td>1996/07/01 1800 </td>,
 <td> 1996/07/02 1100 </td>,
 <td>... </td>,
 <td>... </td>,
 <td> 0  </td>,
 <td> 0 </td>,
 <td>N </td>,
 <td>... </td>,
 <td> 3 </td>,
 <td>  40  </td>,
 <td>360 </td>,
 <td>  370 </td>,
 <td>  11 </td>,
 <td>  2 </td>,
 <td>         -20 </td>,
 <td> ... </td>,
 <td>    </td>,
 <td>1996/08/07 0600 </td>,
 <td>1996/08/07 1200 </td>,
 <td> 1996/08/08 1000 </td>,
 <td>... </td>,
 <td>... </td>,
 <td>0 </td>,
 <td> 0 </td>,
 <td>N </td>,
 <td>... </td>,
 <td> 2 </td>,
 <td>10  </td>,
 <td>350 </td>,
 <td>  380 </td>,
 <td>   7 </td>,
 <td>  2 </td>,
 <td>         -23 </td>,
 <td> ... </td>,
 <td> </td>,
 <

In [148]:
data_entries[0 : 5]

[<td>1996/05/27 1500 </td>,
 <td>1996/05/27 1500 </td>,
 <td> 1996/05/29 0300 </td>,
 <td>... </td>,
 <td>... </td>]

In [156]:
str(data_entries[3])

'<td>... </td>'

In [158]:
'... ' in str(data_entries[3])

True

In [123]:
%%time

data_separated = []

for enum, ii in enumerate(data_entries[0 : 72]):
    one_entry = []
    print(enum)
    
    if ('-' in ii) and ('\xa0' in ii + 2):
        data_separated.append(data_entries[])
    

SyntaxError: expected ':' (<unknown>, line 7)

In [74]:
data_entries[17], data_entries[35]

(<td> </td>, <td>    </td>)

In [95]:
data_entries[17], str(data_entries[17])

(<td> </td>, '<td>\xa0</td>')

In [93]:
data_entries[34], str(data_entries[34])

(<td> ... </td>, '<td> ... </td>')

In [94]:
data_entries[35], str(data_entries[35])

(<td>    </td>, '<td>   \xa0</td>')

In [115]:
'\xa0' in str(data_entries[35])

True

In [21]:
307 * 38

11666

In [22]:
for enum, ii in enumerate(headers1):
    if len(ii) > 329:
        print(enum, '   ', len(ii))

324     40859
337     39268
387     33239
424     28969
451     25902
473     23400
505     19868
520     18184
531     16889
541     15713
550     14629
556     13830
568     12227
589     9687
618     6105
648     2005


In [23]:
for enum, ii in enumerate(headers1[0 : 325]):
    if len(ii) > 150:
        print(enum)

0
5
28
65
99
151
200
227
250
272
304
318
324


In [47]:
data_entries_edit[0 : 18]

[<td>1996/05/27 1500 </td>,
 <td>1996/05/27 1500 </td>,
 <td> 1996/05/29 0300 </td>,
 <td>... </td>,
 <td>... </td>,
 <td> 0 </td>,
 <td> +4 </td>,
 <td>N </td>,
 <td>... </td>,
 <td> 2 </td>,
 <td>0  </td>,
 <td>370 </td>,
 <td>  400 </td>,
 <td>   9 </td>,
 <td>  2 </td>,
 <td>            -33 </td>,
 <td> ... </td>,
 <td> </td>]

In [59]:
# for index 0 [Disturbance (a)]

ii = data_entries_edit[0]

print(str(ii))
str(ii).replace('<td>', '').replace('</td>', '').strip().split(' ')

# same for index 1 and 2 [ICME Plasma/Field Start, End (b)]

<td>1996/05/27 1500 </td>


['1996/05/27', '1500']

In [60]:
# for index 3

ii = data_entries_edit[3]

print(str(ii))
str(ii).replace('<td>', '').replace('</td>', '').strip().split(' ')

# same for index 4

<td>... </td>


['...']

In [61]:
# for index 9 [Qual (g)]
ii = data_entries_edit[9]
str(ii).replace('<td>', '').replace('</td>', '').strip().split()

['2']

In [63]:
# for index 11 [V_ICME (km/s) (i)]
ii = data_entries_edit[11]
str(ii).replace('<td>', '').replace('</td>', '').strip()

# same for index 12 [V_max (km/s) (i)]
# same for index 13 [B (nT) (k)]
# same for index 14 [MC? (l)]
# same for index 15 [Dst (nT) (m)]
# same for index 16 [V_transit (km/s) (n)]

'370'

In [91]:
# for index 17 [LASCO CME... (UT) (o)]
ii = data_entries_edit[17]
str(ii).replace('<td>', '').replace('</td>', '').split(' ')

['\xa0']

In [93]:
ii = data_entries_edit[71]
str(ii).replace('<td>', '').replace('</td>', '').strip().split(' ')

['1996/12/19', '1630', 'H']

In [ ]:
## range for normal data_rows
ii_range = list(range(0, 103))
ii_range.extend(list(range(104, 123)))
ii_range.extend(list(range(124, 133)))
ii_range.extend(list(range(134, 145)))
ii_range.extend([146])
ii_range.extend(range(149, 152))
ii_range.extend(range(153, 156))
ii_range.extend(range(157, 163))
ii_range.extend([165])
ii_range.extend(range(170, 172))